In [1]:
# ============================================================
# PHASE 2.3 — CROSS-MODALITY ERROR COMPARISON
# Protein vs Genomic regulatory predictions
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix
)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 150)

In [2]:
# ============================================================
# PATHS
# ============================================================
import os
from pathlib import Path
import pandas as pd
from google.colab import drive

try:
    drive.mount('/content/drive')
except ValueError:
    print("Drive đã được kết nối từ trước.")

PROJECT_DIR = Path("/content/drive/MyDrive/Project_Protein")

# Protein branch outputs
PHASE1_ANALYSIS_DIR = PROJECT_DIR / "model" / "phase1_final_protein_analysis"
PHASE1_RESULT_DIR = PHASE1_ANALYSIS_DIR / "results"

# ProtBERT sliding-window direct folder if needed
PROTBERT_SW_DIR = PROJECT_DIR / "model" / "phase1_2_protbert_sliding_window_embedding"

# Genomic branch outputs
PHASE2_DIR = PROJECT_DIR / "model" / "phase2_genomic_regulatory_baseline"
PHASE2_RESULT_DIR = PHASE2_DIR / "results"

PHASE2_2_DIR = PROJECT_DIR / "model" / "phase2_2_genomic_analysis"
PHASE2_2_RESULT_DIR = PHASE2_2_DIR / "results"

# New output folder
PHASE2_3_DIR = PROJECT_DIR / "model" / "phase2_3_cross_modality_error_analysis"
RESULT_DIR = PHASE2_3_DIR / "results"
REPORT_DIR = PHASE2_3_DIR / "reports"

for folder in [PHASE2_3_DIR, RESULT_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Phase 2.3 output:", PHASE2_3_DIR)

Mounted at /content/drive
Phase 2.3 output: /content/drive/MyDrive/Project_Protein/model/phase2_3_cross_modality_error_analysis


In [3]:
# ============================================================
# LOCATE PROTEIN PREDICTION FILES
# ============================================================

candidate_protein_prediction_paths = [
    PHASE1_RESULT_DIR / "error_analysis_protbert_sliding_window_test_predictions.csv",
    PROTBERT_SW_DIR / "results" / "protbert_sw_final_test_predictions_v1.csv",
    PROTBERT_SW_DIR / "results" / "protbert_final_test_predictions_v1.csv",
]

for p in candidate_protein_prediction_paths:
    print(p, "exists:", p.exists())

/content/drive/MyDrive/Project_Protein/model/phase1_final_protein_analysis/results/error_analysis_protbert_sliding_window_test_predictions.csv exists: True
/content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/results/protbert_sw_final_test_predictions_v1.csv exists: True
/content/drive/MyDrive/Project_Protein/model/phase1_2_protbert_sliding_window_embedding/results/protbert_final_test_predictions_v1.csv exists: False


In [4]:
# ============================================================
# LOAD PROTEIN PREDICTIONS
# ============================================================

protein_pred_path = None

for p in candidate_protein_prediction_paths:
    if p.exists():
        protein_pred_path = p
        break

assert protein_pred_path is not None, "No protein prediction file found. Check paths."

protein_pred_df = pd.read_csv(protein_pred_path)

print("Loaded protein predictions:", protein_pred_path)
print("Shape:", protein_pred_df.shape)
print("Columns:")
print(protein_pred_df.columns.tolist())

display(protein_pred_df.head())

Loaded protein predictions: /content/drive/MyDrive/Project_Protein/model/phase1_final_protein_analysis/results/error_analysis_protbert_sliding_window_test_predictions.csv
Shape: (273, 16)
Columns:
['row_index', 'gene_id', 'gene_symbol', 'uniprot_accession', 'label', 'sequence_clean_length', 'n_windows', 'window_size', 'stride', 'representation', 'true_label', 'pred_score', 'pred_label', 'threshold', 'model_name', 'error_type']


,row_index,gene_id,gene_symbol,uniprot_accession,label,sequence_clean_length,n_windows,window_size,stride,representation,true_label,pred_score,pred_label,threshold,model_name,error_type
0,0,ENSG00000177542,SLC25A22,Q9H936,0,323,1,1022,1022,ProtBERT_sliding_window_1022_stride_1022,0,0.758499,1,0.425,Logistic Regression,FP
1,1,ENSG00000123080,CDKN2C,P42773,1,168,1,1022,1022,ProtBERT_sliding_window_1022_stride_1022,1,0.464095,1,0.425,Logistic Regression,TP
2,2,ENSG00000185262,UBALD2,Q8IYN6,0,164,1,1022,1022,ProtBERT_sliding_window_1022_stride_1022,0,0.806849,1,0.425,Logistic Regression,FP
3,3,ENSG00000092148,HECTD1,Q9ULT8,0,2610,3,1022,1022,ProtBERT_sliding_window_1022_stride_1022,0,0.613123,1,0.425,Logistic Regression,FP
4,4,ENSG00000139364,TMEM132B,Q14DG7,1,1078,2,1022,1022,ProtBERT_sliding_window_1022_stride_1022,1,0.493005,1,0.425,Logistic Regression,TP


In [5]:
# ============================================================
# STANDARDIZE PROTEIN PREDICTION TABLE
# ============================================================

def find_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


protein_gene_id_col = find_col(protein_pred_df, ["gene_id", "ensembl_gene_id"])
protein_gene_symbol_col = find_col(protein_pred_df, ["gene_symbol", "ensembl_gene_name"])
protein_label_col = find_col(protein_pred_df, ["true_label", "label"])
protein_score_col = find_col(protein_pred_df, [
    "pred_score_t2d_associated",
    "pred_score",
    "score",
    "probability",
    "positive_score"
])

assert protein_gene_id_col is not None, "Protein gene_id column not found."
assert protein_gene_symbol_col is not None, "Protein gene_symbol column not found."
assert protein_label_col is not None, "Protein label column not found."
assert protein_score_col is not None, "Protein prediction score column not found."

protein_std_df = protein_pred_df[[
    protein_gene_id_col,
    protein_gene_symbol_col,
    protein_label_col,
    protein_score_col
]].copy()

protein_std_df = protein_std_df.rename(columns={
    protein_gene_id_col: "gene_id",
    protein_gene_symbol_col: "gene_symbol_protein",
    protein_label_col: "true_label_protein",
    protein_score_col: "protein_score"
})

protein_std_df["true_label_protein"] = protein_std_df["true_label_protein"].astype(int)
protein_std_df["protein_score"] = protein_std_df["protein_score"].astype(float)

# Two protein threshold policies
PROTEIN_DEFAULT_THRESHOLD = 0.5
PROTEIN_TUNED_F1_THRESHOLD = 0.348

protein_std_df["protein_pred_default_0_5"] = (
    protein_std_df["protein_score"] >= PROTEIN_DEFAULT_THRESHOLD
).astype(int)

protein_std_df["protein_pred_tuned_f1"] = (
    protein_std_df["protein_score"] >= PROTEIN_TUNED_F1_THRESHOLD
).astype(int)

display(protein_std_df.head())
print(protein_std_df.shape)

,gene_id,gene_symbol_protein,true_label_protein,protein_score,protein_pred_default_0_5,protein_pred_tuned_f1
0,ENSG00000177542,SLC25A22,0,0.758499,1,1
1,ENSG00000123080,CDKN2C,1,0.464095,0,1
2,ENSG00000185262,UBALD2,0,0.806849,1,1
3,ENSG00000092148,HECTD1,0,0.613123,1,1
4,ENSG00000139364,TMEM132B,1,0.493005,0,1


(273, 6)


In [6]:
# ============================================================
# LOCATE GENOMIC PREDICTION FILES
# ============================================================

candidate_genomic_prediction_paths = [
    PHASE2_2_RESULT_DIR / "phase2_2_error_analysis_random_forest_test_predictions.csv",
    PHASE2_RESULT_DIR / "phase2_1_genomic_final_test_predictions_v1.csv",
]

for p in candidate_genomic_prediction_paths:
    print(p, "exists:", p.exists())

/content/drive/MyDrive/Project_Protein/model/phase2_2_genomic_analysis/results/phase2_2_error_analysis_random_forest_test_predictions.csv exists: True
/content/drive/MyDrive/Project_Protein/model/phase2_genomic_regulatory_baseline/results/phase2_1_genomic_final_test_predictions_v1.csv exists: True


In [7]:
# ============================================================
# LOAD GENOMIC PREDICTIONS
# ============================================================

genomic_pred_path = None

for p in candidate_genomic_prediction_paths:
    if p.exists():
        genomic_pred_path = p
        break

assert genomic_pred_path is not None, "No genomic prediction file found. Check paths."

genomic_pred_df = pd.read_csv(genomic_pred_path)

print("Loaded genomic predictions:", genomic_pred_path)
print("Shape:", genomic_pred_df.shape)
print("Columns:")
print(genomic_pred_df.columns.tolist())

display(genomic_pred_df.head())

Loaded genomic predictions: /content/drive/MyDrive/Project_Protein/model/phase2_2_genomic_analysis/results/phase2_2_error_analysis_random_forest_test_predictions.csv
Shape: (271, 27)
Columns:
['gene_id', 'gene_symbol', 'gene_name', 'label', 'dataset_role', 'ensembl_gene_id', 'ensembl_gene_name', 'gene_type', 'chromosome_or_scaffold', 'gene_start_bp', 'gene_end_bp', 'strand', 'tss_sequence_orientation', 'regulatory_sequence_source', 'regulatory_sequence_scope', 'regulatory_sequence_version', 'regulatory_sequence', 'regulatory_sequence_length', 'n_N_bases', 'invalid_dna_chars', 'true_label', 'pred_score_t2d_associated', 'pred_label', 'threshold', 'model', 'feature_set', 'error_type']


,gene_id,gene_symbol,gene_name,label,dataset_role,ensembl_gene_id,ensembl_gene_name,gene_type,chromosome_or_scaffold,gene_start_bp,gene_end_bp,strand,tss_sequence_orientation,regulatory_sequence_source,regulatory_sequence_scope,regulatory_sequence_version,regulatory_sequence,regulatory_sequence_length,n_N_bases,invalid_dna_chars,true_label,pred_score_t2d_associated,pred_label,threshold,model,feature_set,error_type
0,ENSG00000101639,CEP192,centrosomal protein 192,1,positive,ENSG00000101639,CEP192,protein_coding,18,12991315.0,13125053.0,1.0,forward_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,GTATAAGATAATAGCTAATGTTTGTTGAGTATTTCCATGGGCCATTATTTTTCTAGATACCTTACAGAAGTAGCTTTAAATCCTCACAGTAAGCCTCTGAGGCAAGCGCTTTAATTATCCCCACTTTACAGACAAGGAAACCGGCT...,2500,0,NaN,1,0.452821,0,0.6,Random Forest,K3 + K4 + Basic,FN
1,ENSG00000198793,MTOR,mechanistic target of rapamycin kinase,1,positive,ENSG00000198793,MTOR,protein_coding,1,11106531.0,11262556.0,-1.0,reverse_complement_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,TATTATACTTACATTGTTGCACAACTTGTAATTTTTAAACTAGTTGTGAACCTTTTCCTTCTTAGCAAATATTCATCTAATTACCACTATTTATTGTTATTATTATTATTTGAGACAGGATCTTTCTCTGCTGCCCAGGCTGCAGT...,2500,0,NaN,1,0.386692,0,0.6,Random Forest,K3 + K4 + Basic,FN
2,ENSG00000064607,SUGP2,SUGP2,0,negative,ENSG00000064607,SUGP2,protein_coding,19,18990888.0,19034040.0,-1.0,reverse_complement_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,AGTACAGATTTGGAACACATAAGATGGTGCCTGGAGATGGCCAAGTAAGAATTACCTGGAGAACAGCCTAGGCTAAAAGGGGGAAGCAGCATTACCAGTCCCTATTATCCATCAGGGCATGCTGAGGGTAAAACAGAAGATGAGTA...,2500,0,NaN,0,0.490883,0,0.6,Random Forest,K3 + K4 + Basic,TN
3,ENSG00000180332,KCTD4,KCTD4,0,negative,ENSG00000180332,KCTD4,protein_coding,13,45192853.0,45201045.0,-1.0,reverse_complement_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,AGTATCCTATATTCAAAGCAAAGGCCTTGTCTTACATATACATCAGCAACTAAATGTACATTTATTTTATTTTATTTTATTTCTGAGATGATGTCTCACTCTGTCACCCAGGCTGGAGTAGCTGGAGTACAGTGGTGCAATCTCAA...,2500,0,NaN,0,0.460903,0,0.6,Random Forest,K3 + K4 + Basic,TN
4,ENSG00000166226,CCT2,CCT2,0,negative,ENSG00000166226,CCT2,protein_coding,12,69585423.0,69601577.0,1.0,forward_gene_orientation,Ensembl Homo sapiens GRCh38 primary assembly FASTA,TSS-proximal window: 2000 bp upstream + 500 bp downstream,TSS_Proximal_2kbUp_500bpDown_NuclearBalanced_v1,CATTGGTTTCCCCATCACACACAGTCTCAGGCTTGCCTACCAGAGGACATAATAAATATCTCCAGGTATGTAATTTTGCCCACAATAAAAACAACCTTTTTCAATATTGTGCAGGCATCTTAGTTTTTTATTGTGAATCTTCTCAT...,2500,0,NaN,0,0.392621,0,0.6,Random Forest,K3 + K4 + Basic,TN


In [8]:
# ============================================================
# STANDARDIZE GENOMIC PREDICTION TABLE
# ============================================================

genomic_gene_id_col = find_col(genomic_pred_df, ["gene_id", "ensembl_gene_id"])
genomic_gene_symbol_col = find_col(genomic_pred_df, ["gene_symbol", "ensembl_gene_name"])
genomic_label_col = find_col(genomic_pred_df, ["true_label", "label"])
genomic_score_col = find_col(genomic_pred_df, [
    "pred_score_t2d_associated",
    "pred_score",
    "score",
    "probability",
    "positive_score"
])

assert genomic_gene_id_col is not None, "Genomic gene_id column not found."
assert genomic_gene_symbol_col is not None, "Genomic gene_symbol column not found."
assert genomic_label_col is not None, "Genomic label column not found."
assert genomic_score_col is not None, "Genomic prediction score column not found."

genomic_std_df = genomic_pred_df[[
    genomic_gene_id_col,
    genomic_gene_symbol_col,
    genomic_label_col,
    genomic_score_col
]].copy()

genomic_std_df = genomic_std_df.rename(columns={
    genomic_gene_id_col: "gene_id",
    genomic_gene_symbol_col: "gene_symbol_genomic",
    genomic_label_col: "true_label_genomic",
    genomic_score_col: "genomic_score"
})

genomic_std_df["true_label_genomic"] = genomic_std_df["true_label_genomic"].astype(int)
genomic_std_df["genomic_score"] = genomic_std_df["genomic_score"].astype(float)

# Genomic thresholds
GENOMIC_DEFAULT_THRESHOLD = 0.5
GENOMIC_BALANCED_THRESHOLD = 0.518

genomic_std_df["genomic_pred_default_0_5"] = (
    genomic_std_df["genomic_score"] >= GENOMIC_DEFAULT_THRESHOLD
).astype(int)

genomic_std_df["genomic_pred_balanced_0_518"] = (
    genomic_std_df["genomic_score"] >= GENOMIC_BALANCED_THRESHOLD
).astype(int)

display(genomic_std_df.head())
print(genomic_std_df.shape)

,gene_id,gene_symbol_genomic,true_label_genomic,genomic_score,genomic_pred_default_0_5,genomic_pred_balanced_0_518
0,ENSG00000101639,CEP192,1,0.452821,0,0
1,ENSG00000198793,MTOR,1,0.386692,0,0
2,ENSG00000064607,SUGP2,0,0.490883,0,0
3,ENSG00000180332,KCTD4,0,0.460903,0,0
4,ENSG00000166226,CCT2,0,0.392621,0,0


(271, 6)


In [9]:
# ============================================================
# MERGE PROTEIN + GENOMIC PREDICTIONS
# ============================================================

merged_df = protein_std_df.merge(
    genomic_std_df,
    on="gene_id",
    how="inner"
)

print("Protein predictions:", protein_std_df.shape)
print("Genomic predictions:", genomic_std_df.shape)
print("Merged:", merged_df.shape)

display(merged_df.head())

# Check label consistency
merged_df["label_match"] = (
    merged_df["true_label_protein"] == merged_df["true_label_genomic"]
)

print("Label match counts:")
print(merged_df["label_match"].value_counts())

assert merged_df["label_match"].all(), "Protein and genomic labels do not match."

merged_df["true_label"] = merged_df["true_label_protein"]

# Prefer gene_symbol from genomic if available, fallback protein
merged_df["gene_symbol"] = merged_df["gene_symbol_genomic"].fillna(
    merged_df["gene_symbol_protein"]
)

print("Merged label counts:")
print(merged_df["true_label"].value_counts().sort_index())

Protein predictions: (273, 6)
Genomic predictions: (271, 6)
Merged: (45, 11)


,gene_id,gene_symbol_protein,true_label_protein,protein_score,protein_pred_default_0_5,protein_pred_tuned_f1,gene_symbol_genomic,true_label_genomic,genomic_score,genomic_pred_default_0_5,genomic_pred_balanced_0_518
0,ENSG00000177542,SLC25A22,0,0.758499,1,1,SLC25A22,0,0.444255,0,0
1,ENSG00000076864,RAP1GAP,1,0.661705,1,1,RAP1GAP,1,0.458036,0,0
2,ENSG00000147127,RAB41,0,0.323846,0,0,RAB41,0,0.495373,0,0
3,ENSG00000169783,LINGO1,1,0.763429,1,1,LINGO1,1,0.648438,1,1
4,ENSG00000150676,CCDC83,0,0.319752,0,0,CCDC83,0,0.496439,0,0


Label match counts:
label_match
True    45
Name: count, dtype: int64
Merged label counts:
true_label
0    26
1    19
Name: count, dtype: int64


In [10]:
# ============================================================
# DEFINE MAIN POLICY FOR COMPARISON
# ============================================================

PROTEIN_PRED_COL = "protein_pred_tuned_f1"
GENOMIC_PRED_COL = "genomic_pred_default_0_5"

merged_df["protein_correct"] = (
    merged_df[PROTEIN_PRED_COL] == merged_df["true_label"]
)

merged_df["genomic_correct"] = (
    merged_df[GENOMIC_PRED_COL] == merged_df["true_label"]
)

def assign_cross_modality_group(row):
    if row["protein_correct"] and row["genomic_correct"]:
        return "both_correct"
    elif (not row["protein_correct"]) and (not row["genomic_correct"]):
        return "both_wrong"
    elif row["protein_correct"] and (not row["genomic_correct"]):
        return "protein_correct_genomic_wrong"
    elif (not row["protein_correct"]) and row["genomic_correct"]:
        return "genomic_correct_protein_wrong"
    else:
        return "unknown"

merged_df["cross_modality_group"] = merged_df.apply(
    assign_cross_modality_group,
    axis=1
)

group_counts = merged_df["cross_modality_group"].value_counts().reset_index()
group_counts.columns = ["cross_modality_group", "n"]
group_counts["percentage"] = group_counts["n"] / len(merged_df) * 100

display(group_counts)

group_counts.to_csv(
    RESULT_DIR / "phase2_3_cross_modality_group_counts.csv",
    index=False
)

,cross_modality_group,n,percentage
0,both_correct,15,33.333333
1,genomic_correct_protein_wrong,10,22.222222
2,protein_correct_genomic_wrong,10,22.222222
3,both_wrong,10,22.222222


In [11]:
# ============================================================
# CROSS-MODALITY GROUP BY TRUE LABEL
# ============================================================

group_by_label_df = pd.crosstab(
    merged_df["cross_modality_group"],
    merged_df["true_label"],
    margins=True
)

display(group_by_label_df)

group_by_label_df.to_csv(
    RESULT_DIR / "phase2_3_cross_modality_group_by_label.csv"
)

true_label,0,1,All
cross_modality_group,,,
both_correct,7,8,15
both_wrong,9,1,10
genomic_correct_protein_wrong,9,1,10
protein_correct_genomic_wrong,1,9,10
All,26,19,45


In [12]:
# ============================================================
# FAIR METRICS ON OVERLAPPING GENE SUBSET
# ============================================================

def evaluate_predictions(y_true, y_score, y_pred, model_name):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    return {
        "model": model_name,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_score),
        "pr_auc": average_precision_score(y_true, y_score),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

overlap_metrics = []

overlap_metrics.append(
    evaluate_predictions(
        y_true=merged_df["true_label"],
        y_score=merged_df["protein_score"],
        y_pred=merged_df[PROTEIN_PRED_COL],
        model_name=f"Protein ProtBERT SW LR ({PROTEIN_PRED_COL})"
    )
)

overlap_metrics.append(
    evaluate_predictions(
        y_true=merged_df["true_label"],
        y_score=merged_df["genomic_score"],
        y_pred=merged_df[GENOMIC_PRED_COL],
        model_name=f"Genomic RF K3K4Basic ({GENOMIC_PRED_COL})"
    )
)

overlap_metrics_df = pd.DataFrame(overlap_metrics)

display(overlap_metrics_df)

overlap_metrics_df.to_csv(
    RESULT_DIR / "phase2_3_overlap_subset_modality_metrics.csv",
    index=False
)

,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Protein ProtBERT SW LR (protein_pred_tuned_f1),45,0.555556,0.485714,0.894737,0.307692,0.629630,0.714575,0.670357,0.240493,8,18,2,17
1,Genomic RF K3K4Basic (genomic_pred_default_0_5),45,0.555556,0.473684,0.473684,0.615385,0.473684,0.536437,0.534735,0.089069,16,10,10,9


In [13]:
# ============================================================
# SIMPLE SCORE FUSION DIAGNOSTIC
# Not final integration. Just checks complementarity.
# ============================================================

merged_df["fusion_score_mean"] = (
    merged_df["protein_score"] + merged_df["genomic_score"]
) / 2

# Try default 0.5 for fusion
merged_df["fusion_pred_default_0_5"] = (
    merged_df["fusion_score_mean"] >= 0.5
).astype(int)

fusion_default_metrics = evaluate_predictions(
    y_true=merged_df["true_label"],
    y_score=merged_df["fusion_score_mean"],
    y_pred=merged_df["fusion_pred_default_0_5"],
    model_name="Simple mean fusion score default 0.5"
)

fusion_default_metrics_df = pd.DataFrame([fusion_default_metrics])

display(fusion_default_metrics_df)

fusion_default_metrics_df.to_csv(
    RESULT_DIR / "phase2_3_simple_score_fusion_default_metrics.csv",
    index=False
)

,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Simple mean fusion score default 0.5,45,0.666667,0.590909,0.684211,0.653846,0.634146,0.690283,0.634041,0.334024,17,9,6,13


In [14]:
# ============================================================
# DIAGNOSTIC FUSION THRESHOLD SWEEP ON OVERLAP TEST SUBSET
# Warning: exploratory only, not official final evaluation.
# ============================================================

thresholds = np.arange(0.05, 0.951, 0.001)
fusion_rows = []

for th in thresholds:
    pred = (merged_df["fusion_score_mean"] >= th).astype(int)

    metrics = evaluate_predictions(
        y_true=merged_df["true_label"],
        y_score=merged_df["fusion_score_mean"],
        y_pred=pred,
        model_name="Simple mean fusion"
    )

    metrics["threshold"] = th
    fusion_rows.append(metrics)

fusion_threshold_df = pd.DataFrame(fusion_rows)

best_fusion_mcc = fusion_threshold_df.sort_values(
    by=["mcc", "f1", "roc_auc"],
    ascending=False
).iloc[0]

best_fusion_f1 = fusion_threshold_df.sort_values(
    by=["f1", "mcc", "roc_auc"],
    ascending=False
).iloc[0]

print("Best fusion by MCC:")
display(pd.DataFrame([best_fusion_mcc]))

print("Best fusion by F1:")
display(pd.DataFrame([best_fusion_f1]))

fusion_threshold_df.to_csv(
    RESULT_DIR / "phase2_3_simple_fusion_threshold_sweep_diagnostic.csv",
    index=False
)

Best fusion by MCC:


,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,threshold
456,Simple mean fusion,45,0.688889,0.619048,0.684211,0.692308,0.65,0.690283,0.634041,0.372764,18,8,6,13,0.506


Best fusion by F1:


,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,threshold
330,Simple mean fusion,45,0.577778,0.5,1.0,0.269231,0.666667,0.690283,0.634041,0.3669,7,19,0,19,0.38


In [15]:
# ============================================================
# SAVE MERGED CROSS-MODALITY TABLE
# ============================================================

columns_to_save = [
    "gene_id",
    "gene_symbol",
    "true_label",
    "protein_score",
    "genomic_score",
    "fusion_score_mean",
    "protein_pred_default_0_5",
    "protein_pred_tuned_f1",
    "genomic_pred_default_0_5",
    "genomic_pred_balanced_0_518",
    "fusion_pred_default_0_5",
    "protein_correct",
    "genomic_correct",
    "cross_modality_group"
]

available_cols = [col for col in columns_to_save if col in merged_df.columns]

cross_modality_table = merged_df[available_cols].copy()

display(cross_modality_table.head())

cross_modality_table.to_csv(
    RESULT_DIR / "phase2_3_cross_modality_merged_predictions.csv",
    index=False
)

print("Saved merged cross-modality table.")

,gene_id,gene_symbol,true_label,protein_score,genomic_score,fusion_score_mean,protein_pred_default_0_5,protein_pred_tuned_f1,genomic_pred_default_0_5,genomic_pred_balanced_0_518,fusion_pred_default_0_5,protein_correct,genomic_correct,cross_modality_group
0,ENSG00000177542,SLC25A22,0,0.758499,0.444255,0.601377,1,1,0,0,1,False,True,genomic_correct_protein_wrong
1,ENSG00000076864,RAP1GAP,1,0.661705,0.458036,0.559871,1,1,0,0,1,True,False,protein_correct_genomic_wrong
2,ENSG00000147127,RAB41,0,0.323846,0.495373,0.409610,0,0,0,0,0,True,True,both_correct
3,ENSG00000169783,LINGO1,1,0.763429,0.648438,0.705934,1,1,1,1,1,True,True,both_correct
4,ENSG00000150676,CCDC83,0,0.319752,0.496439,0.408096,0,0,0,0,0,True,True,both_correct


Saved merged cross-modality table.


In [16]:
# ============================================================
# EXPORT GROUP-SPECIFIC TABLES
# ============================================================

for group_name, group_df in merged_df.groupby("cross_modality_group"):
    out_path = RESULT_DIR / f"phase2_3_group_{group_name}.csv"

    group_export = group_df[available_cols].copy()
    group_export = group_export.sort_values(
        by=["true_label", "protein_score", "genomic_score"],
        ascending=[False, False, False]
    )

    group_export.to_csv(out_path, index=False)

    print(group_name, group_export.shape, "saved:", out_path)

both_correct (15, 14) saved: /content/drive/MyDrive/Project_Protein/model/phase2_3_cross_modality_error_analysis/results/phase2_3_group_both_correct.csv
both_wrong (10, 14) saved: /content/drive/MyDrive/Project_Protein/model/phase2_3_cross_modality_error_analysis/results/phase2_3_group_both_wrong.csv
genomic_correct_protein_wrong (10, 14) saved: /content/drive/MyDrive/Project_Protein/model/phase2_3_cross_modality_error_analysis/results/phase2_3_group_genomic_correct_protein_wrong.csv
protein_correct_genomic_wrong (10, 14) saved: /content/drive/MyDrive/Project_Protein/model/phase2_3_cross_modality_error_analysis/results/phase2_3_group_protein_correct_genomic_wrong.csv


In [17]:
# ============================================================
# TOP DISAGREEMENT CASES
# ============================================================

# Protein high, genomic low
protein_high_genomic_low = merged_df.copy()
protein_high_genomic_low["score_gap_protein_minus_genomic"] = (
    protein_high_genomic_low["protein_score"] - protein_high_genomic_low["genomic_score"]
)

protein_high_genomic_low = protein_high_genomic_low.sort_values(
    by="score_gap_protein_minus_genomic",
    ascending=False
)

display(
    protein_high_genomic_low[[
        "gene_id",
        "gene_symbol",
        "true_label",
        "protein_score",
        "genomic_score",
        "score_gap_protein_minus_genomic",
        "cross_modality_group"
    ]].head(30)
)

protein_high_genomic_low.to_csv(
    RESULT_DIR / "phase2_3_top_protein_high_genomic_low.csv",
    index=False
)

# Genomic high, protein low
genomic_high_protein_low = merged_df.copy()
genomic_high_protein_low["score_gap_genomic_minus_protein"] = (
    genomic_high_protein_low["genomic_score"] - genomic_high_protein_low["protein_score"]
)

genomic_high_protein_low = genomic_high_protein_low.sort_values(
    by="score_gap_genomic_minus_protein",
    ascending=False
)

display(
    genomic_high_protein_low[[
        "gene_id",
        "gene_symbol",
        "true_label",
        "protein_score",
        "genomic_score",
        "score_gap_genomic_minus_protein",
        "cross_modality_group"
    ]].head(30)
)

genomic_high_protein_low.to_csv(
    RESULT_DIR / "phase2_3_top_genomic_high_protein_low.csv",
    index=False
)

,gene_id,gene_symbol,true_label,protein_score,genomic_score,score_gap_protein_minus_genomic,cross_modality_group
17,ENSG00000007314,SCN4A,1,0.816374,0.415789,0.400585,protein_correct_genomic_wrong
0,ENSG00000177542,SLC25A22,0,0.758499,0.444255,0.314244,genomic_correct_protein_wrong
35,ENSG00000100170,SLC5A1,1,0.726727,0.482693,0.244034,protein_correct_genomic_wrong
42,ENSG00000164100,NDST3,0,0.648668,0.432827,0.215841,genomic_correct_protein_wrong
15,ENSG00000119013,NDUFB3,1,0.756751,0.546499,0.210252,both_correct
1,ENSG00000076864,RAP1GAP,1,0.661705,0.458036,0.203669,protein_correct_genomic_wrong
40,ENSG00000087299,L2HGDH,1,0.623358,0.426410,0.196948,protein_correct_genomic_wrong
6,ENSG00000107249,GLIS3,1,0.784121,0.602261,0.181860,both_correct
27,ENSG00000110881,ASIC1,0,0.750606,0.573807,0.176799,both_wrong
8,ENSG00000125970,RALY,1,0.722689,0.547252,0.175437,both_correct


,gene_id,gene_symbol,true_label,protein_score,genomic_score,score_gap_genomic_minus_protein,cross_modality_group
9,ENSG00000087448,KLHL42,1,0.326462,0.700530,0.374067,genomic_correct_protein_wrong
39,ENSG00000163535,SGO2,0,0.182707,0.474079,0.291372,both_correct
28,ENSG00000173679,OR1L1,0,0.230406,0.512183,0.281777,protein_correct_genomic_wrong
21,ENSG00000182195,LDOC1,0,0.246184,0.463055,0.216871,both_correct
14,ENSG00000166813,KIF7,0,0.256099,0.449723,0.193625,both_correct
41,ENSG00000255855,KDM4F,0,0.374863,0.554808,0.179945,both_wrong
11,ENSG00000099715,PCDH11Y,0,0.374384,0.552518,0.178134,both_wrong
4,ENSG00000150676,CCDC83,0,0.319752,0.496439,0.176687,both_correct
2,ENSG00000147127,RAB41,0,0.323846,0.495373,0.171527,both_correct
24,ENSG00000101997,CCDC22,0,0.255134,0.419745,0.164610,both_correct


In [18]:
# ============================================================
# WRITE SUMMARY REPORT
# ============================================================

both_correct_n = int((merged_df["cross_modality_group"] == "both_correct").sum())
both_wrong_n = int((merged_df["cross_modality_group"] == "both_wrong").sum())
protein_only_n = int((merged_df["cross_modality_group"] == "protein_correct_genomic_wrong").sum())
genomic_only_n = int((merged_df["cross_modality_group"] == "genomic_correct_protein_wrong").sum())

summary_text = f"""
# Phase 2.3 — Cross-Modality Error Comparison Summary

## Objective

This analysis compared protein-based and genomic regulatory-based predictions on the overlapping test genes.

The goal was to determine whether protein sequence and TSS-proximal regulatory DNA models make complementary errors.

## Models Compared

Protein model:
- ProtBERT sliding-window + Logistic Regression
- Protein score threshold policy: {PROTEIN_PRED_COL}

Genomic model:
- K3 + K4 + Basic genomic features + Random Forest
- Genomic score threshold policy: {GENOMIC_PRED_COL}

## Overlapping Test Set

Number of overlapping genes: {len(merged_df)}

Label counts:
{merged_df['true_label'].value_counts().sort_index().to_string()}

## Cross-Modality Error Groups

- Both correct: {both_correct_n}
- Both wrong: {both_wrong_n}
- Protein correct, genomic wrong: {protein_only_n}
- Genomic correct, protein wrong: {genomic_only_n}

## Interpretation

If the protein-only-correct and genomic-only-correct groups are substantial, this indicates that the two modalities capture complementary biological signals.

Protein sequence may capture coding/protein-level information, while TSS-proximal regulatory DNA captures promoter/regulatory sequence patterns.

This provides direct justification for Phase 3 multimodal integration.

## Diagnostic Simple Fusion

A simple mean score fusion was tested only as an exploratory diagnostic.

Default fusion metrics:
{fusion_default_metrics_df.to_string(index=False)}

This is not considered the final integration model because it was evaluated on the overlapping test subset without a separate validation protocol.

## Next Step

Proceed to Phase 3 multimodal integration using a shared gene split and features from both modalities.

Recommended Phase 3 input:
- ProtBERT sliding-window protein embeddings
- K3 + K4 + Basic genomic regulatory features

Recommended model families:
- Logistic Regression
- SVM RBF
- Random Forest
- LightGBM
- Soft Voting
- Stacking
"""

summary_path = REPORT_DIR / "phase2_3_cross_modality_error_summary.md"

with open(summary_path, "w") as f:
    f.write(summary_text)

print(summary_text)
print("Saved:", summary_path)


# Phase 2.3 — Cross-Modality Error Comparison Summary

## Objective

This analysis compared protein-based and genomic regulatory-based predictions on the overlapping test genes.

The goal was to determine whether protein sequence and TSS-proximal regulatory DNA models make complementary errors.

## Models Compared

Protein model:
- ProtBERT sliding-window + Logistic Regression
- Protein score threshold policy: protein_pred_tuned_f1

Genomic model:
- K3 + K4 + Basic genomic features + Random Forest
- Genomic score threshold policy: genomic_pred_default_0_5

## Overlapping Test Set

Number of overlapping genes: 45

Label counts:
true_label
0    26
1    19

## Cross-Modality Error Groups

- Both correct: 15
- Both wrong: 10
- Protein correct, genomic wrong: 10
- Genomic correct, protein wrong: 10

## Interpretation

If the protein-only-correct and genomic-only-correct groups are substantial, this indicates that the two modalities capture complementary biological signals.

Protein sequenc

In [19]:
print(merged_df.shape)
display(group_counts)
display(group_by_label_df)
display(overlap_metrics_df)
display(fusion_default_metrics_df)
display(pd.DataFrame([best_fusion_mcc]))
display(pd.DataFrame([best_fusion_f1]))

(45, 19)


,cross_modality_group,n,percentage
0,both_correct,15,33.333333
1,genomic_correct_protein_wrong,10,22.222222
2,protein_correct_genomic_wrong,10,22.222222
3,both_wrong,10,22.222222


true_label,0,1,All
cross_modality_group,,,
both_correct,7,8,15
both_wrong,9,1,10
genomic_correct_protein_wrong,9,1,10
protein_correct_genomic_wrong,1,9,10
All,26,19,45


,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Protein ProtBERT SW LR (protein_pred_tuned_f1),45,0.555556,0.485714,0.894737,0.307692,0.629630,0.714575,0.670357,0.240493,8,18,2,17
1,Genomic RF K3K4Basic (genomic_pred_default_0_5),45,0.555556,0.473684,0.473684,0.615385,0.473684,0.536437,0.534735,0.089069,16,10,10,9


,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp
0,Simple mean fusion score default 0.5,45,0.666667,0.590909,0.684211,0.653846,0.634146,0.690283,0.634041,0.334024,17,9,6,13


,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,threshold
456,Simple mean fusion,45,0.688889,0.619048,0.684211,0.692308,0.65,0.690283,0.634041,0.372764,18,8,6,13,0.506


,model,n,accuracy,precision,recall_sensitivity,specificity,f1,roc_auc,pr_auc,mcc,tn,fp,fn,tp,threshold
330,Simple mean fusion,45,0.577778,0.5,1.0,0.269231,0.666667,0.690283,0.634041,0.3669,7,19,0,19,0.38
